# 09.8 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/08-prompt-engineering/solutions.ipynb)

> **About the outputs in this notebook.** They were produced with `llama3.2` (3B) via Ollama on a laptop. Yours will differ: a different model, a different run, even the same model on a different machine. Small local models make more mistakes than large hosted ones; that is part of what this lesson shows.

In [1]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

In [2]:
sys.path.insert(0, str(repo_file("lib/course_llm.py").parent))
import course_llm
from course_llm import ask, ask_json, chat, info
info();

LLM provider: ollama · model: llama3.2


In [3]:
# 20 real-looking ShopWave support emails, labelled by the support team.
EMAILS = [
 ("Hi, I ordered on Monday and the tracking still says 'label created'. When will it arrive?", "delivery"),
 ("My parcel was supposed to come yesterday with express but nothing showed up. Order 58213.", "delivery"),
 ("Do you deliver to Norway, and how long does it take?", "delivery"),
 ("The courier left a card saying they'll try again. Can I pick it up somewhere instead?", "delivery"),
 ("I'd like to send back the linen cushion, it doesn't match my sofa. How do I return it?", "returns_refunds"),
 ("I returned two candles 10 days ago and still haven't seen the money.", "returns_refunds"),
 ("Can I swap the blue throw for the grey one?", "returns_refunds"),
 ("You refunded £24 but I paid £27.95 including delivery. Where's the rest?", "returns_refunds"),
 ("Three of the six wine glasses arrived smashed. Photos attached.", "damaged_item"),
 ("The lamp flickers and then cuts out after a few minutes. It's only two weeks old.", "damaged_item"),
 ("I ordered the large planter but you sent the small one.", "damaged_item"),
 ("The box was soaked and the tea towels inside are stained.", "damaged_item"),
 ("We run a gift shop in York and would like to stock your candles. What are your trade prices?", "wholesale"),
 ("Our café account has had 3 orders now. Can we move to invoice payment?", "wholesale"),
 ("Is there a minimum quantity for wholesale orders of the ceramic mugs?", "wholesale"),
 ("I need 40 matching mugs for our office opening, can we get a business discount?", "wholesale"),
 ("Just wanted to say the fairy lights are gorgeous, thank you!", "other"),
 ("How many Stars do I have and when do they expire?", "other"),
 ("Is the reed diffuser safe to use around cats?", "other"),
 ("Can I change the email address on my account?", "other"),
]
LABELS = ["delivery", "returns_refunds", "damaged_item", "wholesale", "other"]
print(len(EMAILS), "emails;", {l: sum(lab == l for _, lab in EMAILS) for l in LABELS})

20 emails; {'delivery': 4, 'returns_refunds': 4, 'damaged_item': 4, 'wholesale': 4, 'other': 4}


In [4]:
SCHEMA = {"type": "object",
          "properties": {"category": {"type": "string", "enum": LABELS}},
          "required": ["category"], "additionalProperties": False}

import json

def classify(prompt, system=None):
    """One label, at temperature 0 so the same email always gets the same answer."""
    reply = chat([{"role": "user", "content": prompt}], system=system,
                 max_tokens=50, temperature=0, schema=SCHEMA)
    return json.loads(reply)["category"]

def evaluate(make_prompt, system=None):
    """Classify all 20 emails; return accuracy and the mistakes."""
    mistakes = []
    for text, label in EMAILS:
        predicted = classify(make_prompt(text), system)
        if predicted != label:
            mistakes.append((text[:55], label, predicted))
    return 1 - len(mistakes) / len(EMAILS), mistakes

## Exercise 1

In [5]:
FEW_SHOT_SYSTEM = """You sort ShopWave support emails into exactly one category.

Categories:
- delivery: where an order is, delivery times, couriers, shipping destinations
- returns_refunds: sending items back, exchanges, refund amounts and timing
- damaged_item: broken, faulty, stained or wrong items received
- wholesale: businesses buying in bulk, trade accounts, trade prices, invoices
- other: anything else (praise, product questions, accounts, loyalty Stars)

Examples:
Email: "It's been 6 days and my order hasn't shipped." -> delivery
Email: "The vase you sent has a big crack down the side." -> damaged_item
Email: "Can I get my money back for the doormat? I changed my mind." -> returns_refunds
Email: "We own two florists and want to order your vases monthly." -> wholesale
Email: "Do your candles contain paraffin?" -> other"""

In [6]:
definitions_only = FEW_SHOT_SYSTEM.split("Examples:")[0].strip()

def plain(text):
    return f'Email: "{text}"'

acc_def, mistakes = evaluate(plain, system=definitions_only)
print(f"definitions, no examples: {acc_def:.0%}")
for m in mistakes:
    print("  ", m)

definitions, no examples: 100%


Compare this number with the lesson's zero-shot and few-shot results. In our run, definitions alone already matched few-shot: the definitions are where most of the ambiguity is resolved: they tell the model that an exchange is a return, a wrong item counts as damaged, and a shop asking for prices is wholesale. Examples then cover what the definitions leave unclear.

## Exercise 2

In [7]:
hard = [
    ("Great, another week and still no sign of my 'next-day' delivery. Fantastic service.", "delivery"),
    ("Mug arrived chipped. Don't want a replacement, just my money back.", "damaged_item"),
    ("refund?", "returns_refunds"),
    ("We're a hotel and want 200 tea towels, but the last batch had stains.", "wholesale"),
    ("Can I return the cushion I got as a gift without the giver finding out?", "returns_refunds"),
]
for text, label in hard:
    got = classify(f'Email: "{text}"', FEW_SHOT_SYSTEM)
    print(f"{'ok ' if got == label else 'XX '} {label:16} → {got:16} {text[:50]}")

XX  delivery         → other            Great, another week and still no sign of my 'next-


XX  damaged_item     → returns_refunds  Mug arrived chipped. Don't want a replacement, jus


ok  returns_refunds  → returns_refunds  refund?


ok  wholesale        → wholesale        We're a hotel and want 200 tea towels, but the las


ok  returns_refunds  → returns_refunds  Can I return the cushion I got as a gift without t


Sarcasm is a classic miss: *"Fantastic service"* reads as praise if you skim. Mixed emails (a chipped mug *and* a refund request; a trade order *and* a quality complaint) have no single right answer: a person might label them either way. When people disagree, the model cannot be "right"; you need a rule for mixed cases (for example "the most urgent issue wins") written into the prompt.

## Exercise 3

In [8]:
FIELDS = {"type": "object", "properties": {
    "order_number": {"type": "string"}, "product": {"type": "string"},
    "issue": {"type": "string", "enum": ["late", "damaged", "wrong_item", "refund_amount", "other"]},
    "urgent": {"type": "boolean"}},
    "required": ["order_number", "product", "issue", "urgent"], "additionalProperties": False}
for i in [1, 7, 8, 10]:
    text = EMAILS[i][0]
    print(text[:60], "\n   ", ask_json(f"Extract these fields from the email. Use an empty "
          f"string if a field is not mentioned.\n<email>{text}</email>", FIELDS, max_tokens=150))

My parcel was supposed to come yesterday with express but no 
    {'order_number': '58213', 'product': '', 'issue': 'late', 'urgent': False}


You refunded £24 but I paid £27.95 including delivery. Where 
    {'order_number': '', 'product': '', 'issue': 'refund_amount', 'urgent': False}


Three of the six wine glasses arrived smashed. Photos attach 
    {'order_number': '', 'product': 'wine glasses', 'issue': 'damaged', 'urgent': False}


I ordered the large planter but you sent the small one. 
    {'order_number': '', 'product': 'planter', 'issue': 'wrong_item', 'urgent': False}


`order_number`, `product` and `issue` come out right when they are stated plainly, and empty strings are used when there is no order number. `urgent` is a judgement call with no definition: in our run it came back `false` even for the smashed glasses. Define it ("urgent = the customer has waited longer than promised or the item is unusable") if you need it. Always check that an order number was not invented when none was given.

## Exercise 4

Problems with *"write about the candle"*: no audience, no length, no tone, no facts to use (so it invents them), no format, nothing it must not say.

In [9]:
bad = ask("write about the candle", temperature=0, max_tokens=300)
better = ask("Write a product description for ShopWave's online shop.\n"
             "Product facts (use only these): fig and cedar scented candle, 200 g, 40-hour burn.\n"
             "Audience: people buying a gift. Tone: warm, simple, no exaggeration.\n"
             "Format: one paragraph of 40-60 words, then one line 'Burn time: 40 hours'.",
             temperature=0, max_tokens=200)
print(bad[:500], "\n\n----- better prompt -----\n\n", better)

The Candle: A Symbol of Warmth and Comfort

A candle is a simple yet elegant object that has been a part of human culture for thousands of years. Its soft, warm glow has the power to evoke feelings of comfort, relaxation, and tranquility. From ancient rituals to modern-day decor, the candle has played a significant role in our lives, serving as a symbol of warmth, light, and hope.

History of the Candle

The earliest recorded use of candles dates back to ancient Egypt around 3000 BCE. The Egypti 

----- better prompt -----

 Give the gift of warmth and tranquility with our fig and cedar scented candle. Made with high-quality ingredients, this 200g candle is designed to provide a soothing ambiance for any room. Perfect for a thoughtful present, it's sure to be appreciated by anyone who loves a cozy and inviting space.

Burn time: 40 hours.


The bad prompt has to guess which candle, so it invents details. The better prompt supplies the facts, the reader, the tone and the format, and the answer is usable as is.